# Gaze Intersection Error visualization
This notebook contains an interactive visualization of the GIE in a certain frame

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan

# Loading Datasets
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_data_from_disk = Scanned3dEnvironment.from_gathered_robot_data(
    robot_data = robot_data,
    number_of_sampled_datapoints=10,
    sample_datapoints_based_on_aruco_corectness = False,
    only_sample_robot_datapoints_w_marker_estimates = True,
    markers_use_advanced_removal=True,
    est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
    est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

labeled_headset_data = bind_headset_recording_to_scan(headset_data = HeadsetRecording.from_vrs_file(vrs_file_location), robot_data = robot_data)

# Define the localizer
ellipsoids_light_glue = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx = labeled_headset_data.intrinsic_cam_mtx,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(contamination=0.01, size_penalty=0.997, size_p_norm=1),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
        visualize_environment_generation = False,
        visualize_pne_optimisation = False,
        visualize_matching=False
    ),
    name="Ellipsoids"
)

# Grade on whole dataset to get the predicted poses
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[ellipsoids_light_glue],
    headset_data = labeled_headset_data,
    robot_env = robot_data_from_disk,
    compute_ray_intersection_error=True,
)
grader.print_summary()

init_predictor_grade = grader.graders[0]

### Plot the errors as image

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

#Choose index of the visualized frame
INDEX = 40

img = cv2.cvtColor(labeled_headset_data.bgr_image_s[INDEX],cv2.COLOR_BGR2RGB)
pose = [(pred, gt) for i, pred, gt in init_predictor_grade.comparable_poses if i == INDEX]
if len(pose) < 1:
    raise Exception("Pose not comparable")

pred, gt = pose[0]
fge = FastRayIntersectionError(points=robot_data_from_disk.robot_xyz_images, intrinsics=labeled_headset_data.intrinsic_cam_mtx)

mat = fge.compute_ray_intersection_error_img(base_t_cam1=pred, base_t_cam2=gt, dim=(img.shape[1], img.shape[0]), size=3)
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(mat)
plt.colorbar(im)

### Interactive window
The type of image being displayed by the notebook is important, should be interactive javascript (can be changed).
Click in the image to get the error at this position displayed.

In [ ]:
%matplotlib widget
from IPython.display import display, clear_output
import ipywidgets as widgets

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()

clicked_points = []

def on_click(event):
    if event.inaxes != ax:
        return
    
    x, y = int(event.xdata), int(event.ydata)
    
    if 0 <= x < img.shape[1] and 0 <= y < img.shape[0]:
        pixels = sample_pixel_neighborhood(center=(x, y), size=5)
        error = fge.calculate_gripping_differences_4_pixels(
            base_t_cam_s=np.array([pred, gt]),
            pixels_batch=np.array([pixels, pixels])
        )[0, 1]
        
        clicked_points.append(((x, y), error))
        
        ax.plot(x, y, 'ro', markersize=8, markeredgecolor='white', markeredgewidth=2)
        ax.annotate(f"{error*1000:.1f}mm", (x, y), xytext=(10, 10),
                   textcoords='offset points',
                   bbox=dict(boxstyle="round,pad=0.5", fc='yellow', alpha=0.8))
        fig.canvas.draw()
    
fig.canvas.mpl_connect('button_press_event', on_click)

plt.show()